In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import warnings

import nltk
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option('display.max_rows', None)
nltk.download('stopwords', quiet=True)
stop_words_pt = stopwords.words('portuguese')

print("Carregando o modelo Sentence Transformers (isso pode levar alguns segundos)...")
modelo_ia = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Modelo de IA pronto para uso!")

Carregando o modelo Sentence Transformers (isso pode levar alguns segundos)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5684.23it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo de IA pronto para uso!


In [12]:
def executar_experimento_asag(nome_dataset):
    print(f"\n" + "="*70)
    print(f"INICIANDO PIPELINE PARA: {nome_dataset}")
    print("="*70)
    
    PASTA = "dataset"
    
    caminho_train = os.path.join(PASTA, f"{nome_dataset}_train.csv")
    caminho_test  = os.path.join(PASTA, f"{nome_dataset}_test.csv")
    
    if not os.path.exists(caminho_train) or not os.path.exists(caminho_test):
        print(f"Arquivos base não encontrados para {nome_dataset}.")
        return None

    df_train = pd.read_csv(caminho_train)
    df_test  = pd.read_csv(caminho_test)

    textos_train = df_train["answer_text"].fillna("").astype(str).tolist()
    textos_test  = df_test["answer_text"].fillna("").astype(str).tolist()
    
    print(" -> Gerando Embeddings e TF-IDF...")
    emb_train = modelo_ia.encode(textos_train, show_progress_bar=False)
    emb_test  = modelo_ia.encode(textos_test,  show_progress_bar=False)
    
    cols_emb = [f"emb_{i}" for i in range(emb_train.shape[1])]
    df_emb_train = pd.DataFrame(emb_train, columns=cols_emb)
    df_emb_test  = pd.DataFrame(emb_test,  columns=cols_emb)

    tfidf_vec = TfidfVectorizer(max_features=1000, stop_words=stop_words_pt)
    df_tfidf_train = pd.DataFrame(tfidf_vec.fit_transform(textos_train).toarray(), columns=tfidf_vec.get_feature_names_out())
    df_tfidf_test  = pd.DataFrame(tfidf_vec.transform(textos_test).toarray(),  columns=tfidf_vec.get_feature_names_out())
    arquivo_coh_train = os.path.join(PASTA, f"cohmetrix_{nome_dataset}_train.csv")
    arquivo_coh_test  = os.path.join(PASTA, f"cohmetrix_{nome_dataset}_test.csv")

    usar_cohmetrix = os.path.exists(arquivo_coh_train) and os.path.exists(arquivo_coh_test)
    
    if usar_cohmetrix:
        print(" -> Carregando características do Coh-Metrix...")
        coh_train_raw = pd.read_csv(arquivo_coh_train)
        coh_test_raw  = pd.read_csv(arquivo_coh_test)
        
        def limpar_para_merge(texto):
            return re.sub(r'\s+', ' ', str(texto).lower()).strip()

        df_train['chave_merge'] = df_train['answer_text'].apply(limpar_para_merge)
        df_test['chave_merge']  = df_test['answer_text'].apply(limpar_para_merge)
        
        coh_train_raw['chave_merge'] = coh_train_raw['resposta_original'].apply(limpar_para_merge)
        coh_test_raw['chave_merge']  = coh_test_raw['resposta_original'].apply(limpar_para_merge)
        
        coh_train_dedup = coh_train_raw.drop_duplicates(subset='chave_merge')
        coh_test_dedup  = coh_test_raw.drop_duplicates(subset='chave_merge')
        
        # O LEFT JOIN preserva 100% das linhas de treino e teste originais
        df_train_coh = df_train.merge(coh_train_dedup, on='chave_merge', how='left')
        df_test_coh  = df_test.merge(coh_test_dedup, on='chave_merge', how='left')
        
        colunas_remover = ['resposta_original', 'nota_original', 'answer_text', 'grade', 'chave_merge']
        colunas_features = [c for c in coh_train_dedup.columns if c not in colunas_remover]
        
        coh_train = df_train_coh[colunas_features].fillna(0)
        coh_test  = df_test_coh[colunas_features].fillna(0)
        
        print(f"    * Treino: {len(coh_train)} amostras alinhadas perfeitamente.")
        print(f"    * Teste:  {len(coh_test)} amostras alinhadas perfeitamente.")
    else:
        print(" -> AVISO: Arquivos do Coh-Metrix não encontrados. Rodando sem métricas.")

    print(" -> Preparando cenários de características...")
    y_train = df_train["grade"]
    y_test = df_test["grade"]

    if usar_cohmetrix:
        cenarios = {
            "Apenas TF-IDF": (df_tfidf_train, df_tfidf_test),
            "Apenas Embeddings": (df_emb_train, df_emb_test),
            "Apenas Coh-Metrix": (coh_train, coh_test),
            "TF-IDF + Embeddings": (pd.concat([df_tfidf_train, df_emb_train], axis=1), pd.concat([df_tfidf_test, df_emb_test], axis=1)),
            "TF-IDF + Coh-Metrix": (pd.concat([df_tfidf_train, coh_train], axis=1), pd.concat([df_tfidf_test, coh_test], axis=1)),
            "Coh-Metrix + Embeddings": (pd.concat([coh_train, df_emb_train], axis=1), pd.concat([coh_test, df_emb_test], axis=1)),
            "Tudo (TF-IDF + Coh-Metrix + Emb)": (pd.concat([df_tfidf_train, coh_train, df_emb_train], axis=1), pd.concat([df_tfidf_test, coh_test, df_emb_test], axis=1)),
        }
    else:
        cenarios = {
            "Apenas TF-IDF": (df_tfidf_train, df_tfidf_test),
            "Apenas Embeddings": (df_emb_train, df_emb_test),
            "TF-IDF + Embeddings": (pd.concat([df_tfidf_train, df_emb_train], axis=1), pd.concat([df_tfidf_test, df_emb_test], axis=1)),
        }

    print(" -> Treinando e avaliando modelos...")
    modelos = {
        "Regressão Linear": LinearRegression(),
        "KNN": KNeighborsRegressor(n_neighbors=5),
        "SVR": SVR(),
        "Árvore de Decisão": DecisionTreeRegressor(random_state=42),
        "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
        "HistGradientBoosting": HistGradientBoostingRegressor(random_state=42),
        "Rede Neural (MLP)": MLPRegressor(random_state=42, max_iter=1000),
    }

    resultados = []
    melhor_r2 = -float('inf')
    dados_melhor_modelo = {}

    for nome_cenario, (X_train_c, X_test_c) in cenarios.items():
        scaler = StandardScaler()
        colunas = X_train_c.columns
        X_tr_sc = pd.DataFrame(scaler.fit_transform(X_train_c), columns=colunas)
        X_te_sc = pd.DataFrame(scaler.transform(X_test_c), columns=colunas)

        for nome_modelo, modelo_base in modelos.items():
            modelo = clone(modelo_base)
            modelo.fit(X_tr_sc, y_train) 
            y_pred = modelo.predict(X_te_sc)

            mae = mean_absolute_error(y_test, y_pred)
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            r2 = r2_score(y_test, y_pred)

            resultados.append({
                "Dataset": nome_dataset,
                "Cenário": nome_cenario,
                "Modelo": nome_modelo,
                "MAE": mae,
                "RMSE": rmse,
                "R²": r2
            })

            if r2 > melhor_r2:
                melhor_r2 = r2
                dados_melhor_modelo = {
                    'modelo': modelo,
                    'nome_modelo': nome_modelo,
                    'cenario': nome_cenario,
                    'y_pred': y_pred,
                    'y_test': y_test,
                    'df_ref': df_test
                }

    df_resultados = pd.DataFrame(resultados).sort_values(by="R²", ascending=False).reset_index(drop=True)
    df_resultados.to_csv(f"resultados_{nome_dataset}.csv", index=False)
    
    print(f"\nConcluído! Resultados oficiais para {nome_dataset}:")
    display(df_resultados.head(10).round(4))
    print(f"Melhor Modelo: {dados_melhor_modelo['nome_modelo']} ({dados_melhor_modelo['cenario']}) | R²: {melhor_r2:.4f}\n")

    return df_resultados

## 2. Execução em Lote (Batch Processing)
Defina a lista de datasets (questões) a serem processados. O script avaliará cada um individualmente e compilará um relatório geral no final.

In [13]:

lista_de_datasets = ["df_11", "df_12"]

todas_as_tabelas = []

print("Iniciando a bateria de experimentos...")

for dataset in lista_de_datasets:
    try:
        tabela_resultado = executar_experimento_asag(dataset)
        
        if tabela_resultado is not None and not tabela_resultado.empty:
            todas_as_tabelas.append(tabela_resultado)
            
    except Exception as e:
        print(f"Erro crítico ao processar {dataset}: {e}")

if len(todas_as_tabelas) > 0:
    relatorio_final = pd.concat(todas_as_tabelas, ignore_index=True)

    print("\n" + "="*70)
    print("RANKING GERAL DE TODOS OS EXPERIMENTOS (Top 20)")
    print("="*70)
    display(relatorio_final.sort_values(by="R²", ascending=False).head(20).round(4))

    relatorio_final.to_csv("relatorio_geral_experimentos_asag.csv", index=False)
    print("Relatório geral salvo como 'relatorio_geral_experimentos_asag.csv'")
else:
    print("\nNenhum resultado foi gerado. Verifique os logs acima para entender onde falhou.")

Iniciando a bateria de experimentos...

INICIANDO PIPELINE PARA: df_11
 -> Gerando Embeddings e TF-IDF...
 -> Carregando características do Coh-Metrix...
    * Treino: 7220 amostras alinhadas perfeitamente.
    * Teste:  2642 amostras alinhadas perfeitamente.
 -> Preparando cenários de características...
 -> Treinando e avaliando modelos...

Concluído! Resultados oficiais para df_11:


,Dataset,Cenário,Modelo,MAE,RMSE,R²
0,df_11,TF-IDF + Coh-Metrix,Random Forest,0.4605,0.6837,0.4109
1,df_11,TF-IDF + Coh-Metrix,HistGradientBoosting,0.4739,0.6961,0.3893
2,df_11,Coh-Metrix + Embeddings,HistGradientBoosting,0.4681,0.6981,0.3858
3,df_11,Tudo (TF-IDF + Coh-Metrix + Emb),HistGradientBoosting,0.4821,0.7153,0.3551
4,df_11,Coh-Metrix + Embeddings,SVR,0.4895,0.7176,0.3510
5,df_11,Tudo (TF-IDF + Coh-Metrix + Emb),Random Forest,0.4842,0.7210,0.3447
6,df_11,Apenas Coh-Metrix,Random Forest,0.5164,0.7289,0.3304
7,df_11,Coh-Metrix + Embeddings,Random Forest,0.4929,0.7318,0.3250
8,df_11,Apenas Coh-Metrix,HistGradientBoosting,0.5201,0.7367,0.3160
9,df_11,Apenas Coh-Metrix,Regressão Linear,0.5286,0.7369,0.3156


Melhor Modelo: Random Forest (TF-IDF + Coh-Metrix) | R²: 0.4109


INICIANDO PIPELINE PARA: df_12
 -> Gerando Embeddings e TF-IDF...
 -> Carregando características do Coh-Metrix...
    * Treino: 6895 amostras alinhadas perfeitamente.
    * Teste:  2967 amostras alinhadas perfeitamente.
 -> Preparando cenários de características...
 -> Treinando e avaliando modelos...

Concluído! Resultados oficiais para df_12:


,Dataset,Cenário,Modelo,MAE,RMSE,R²
0,df_12,Coh-Metrix + Embeddings,SVR,0.4270,0.6161,0.6610
1,df_12,Apenas Embeddings,SVR,0.4261,0.6174,0.6595
2,df_12,Tudo (TF-IDF + Coh-Metrix + Emb),HistGradientBoosting,0.4383,0.6358,0.6390
3,df_12,Coh-Metrix + Embeddings,HistGradientBoosting,0.4405,0.6399,0.6343
4,df_12,Tudo (TF-IDF + Coh-Metrix + Emb),SVR,0.4457,0.6411,0.6329
5,df_12,TF-IDF + Embeddings,HistGradientBoosting,0.4421,0.6454,0.6280
6,df_12,TF-IDF + Embeddings,SVR,0.4486,0.6468,0.6264
7,df_12,Apenas TF-IDF,Random Forest,0.4219,0.6502,0.6225
8,df_12,Tudo (TF-IDF + Coh-Metrix + Emb),Random Forest,0.4586,0.6560,0.6157
9,df_12,TF-IDF + Coh-Metrix,HistGradientBoosting,0.4575,0.6579,0.6135


Melhor Modelo: SVR (Coh-Metrix + Embeddings) | R²: 0.6610


RANKING GERAL DE TODOS OS EXPERIMENTOS (Top 20)


,Dataset,Cenário,Modelo,MAE,RMSE,R²
49,df_12,Coh-Metrix + Embeddings,SVR,0.4270,0.6161,0.6610
50,df_12,Apenas Embeddings,SVR,0.4261,0.6174,0.6595
51,df_12,Tudo (TF-IDF + Coh-Metrix + Emb),HistGradientBoosting,0.4383,0.6358,0.6390
52,df_12,Coh-Metrix + Embeddings,HistGradientBoosting,0.4405,0.6399,0.6343
53,df_12,Tudo (TF-IDF + Coh-Metrix + Emb),SVR,0.4457,0.6411,0.6329
54,df_12,TF-IDF + Embeddings,HistGradientBoosting,0.4421,0.6454,0.6280
55,df_12,TF-IDF + Embeddings,SVR,0.4486,0.6468,0.6264
56,df_12,Apenas TF-IDF,Random Forest,0.4219,0.6502,0.6225
57,df_12,Tudo (TF-IDF + Coh-Metrix + Emb),Random Forest,0.4586,0.6560,0.6157
58,df_12,TF-IDF + Coh-Metrix,HistGradientBoosting,0.4575,0.6579,0.6135


Relatório geral salvo como 'relatorio_geral_experimentos_asag.csv'
